In [1]:
%pip install jupysql pandas matplotlib duckdb-engine
import duckdb
import pandas as pd
import build_geonames_db

%load_ext sql
conn = build_geonames_db.open_or_init_duckdb()
%sql conn --alias duckdb

Note: you may need to restart the kernel to use updated packages.


The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

In [29]:
from pathlib import Path
print(f"DB Size: {Path(build_geonames_db.DUCK_DB_PATH).stat().st_size / 1e9:.2f} GB")
description = conn.execute("""DESCRIBE""").fetchdf()
description = description[['name', 'column_names', 'column_types']]
description.set_index('name', inplace=True)


for table_name in description.index:
    row_count = conn.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
    description.loc[table_name, 'row_count'] = row_count

display(description.style.format({'row_count': '{:_.0f}'}))

DB Size: 1.88 GB


,column_names,column_types,row_count
name,,,
alternateNames,['alternateNameId' 'geonameId' 'isolanguage' 'alternateName' 'isPreferredName' 'isShortName' 'isColloquial' 'isHistoric'],['INTEGER' 'INTEGER' 'VARCHAR' 'VARCHAR' 'BOOLEAN' 'BOOLEAN' 'BOOLEAN' 'BOOLEAN'],18_769_801
countryInfo,['ISO' 'ISO3' 'ISO_Numeric' 'fips' 'Country' 'Capital' 'Area' 'Population' 'Continent' 'tld' 'CurrencyCode' 'CurrencyName' 'Phone' 'Postal_Code_Format' 'Postal_Code_Regex' 'Languages' 'geonameid' 'neighbours' 'EquivalentFipsCode'],['VARCHAR' 'VARCHAR' 'INTEGER' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'FLOAT' 'INTEGER' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'INTEGER' 'VARCHAR' 'VARCHAR'],252
geonames,['geonameId' 'name' 'asciiname' 'alternatenames' 'latitude' 'longitude' 'feature_class' 'feature_code' 'country_code' 'cc2' 'admin1_code' 'admin2_code' 'admin3_code' 'admin4_code' 'admin5_code' 'population' 'elevation' 'dem' 'timezone' 'modification_date'],['INTEGER' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'FLOAT' 'FLOAT' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'BIGINT' 'INTEGER' 'INTEGER' 'VARCHAR' 'DATE'],13_412_879
gnd,['gndUri' 'geonameId'],['VARCHAR' 'INTEGER'],65_617
gndNames,['gndUri' 'name' 'isPreferred'],['VARCHAR' 'VARCHAR' 'BOOLEAN'],132_370
hierarchy,['parentId' 'childId' 'type'],['INTEGER' 'INTEGER' 'VARCHAR'],518_269


In [30]:
%%sql
SELECT gndUri, name, isPreferred, geonameId, feature_class, feature_code, country_code, admin1_code, admin2_code, admin3_code, admin4_code, admin5_code 
FROM gndNames NATURAL JOIN gnd NATURAL JOIN geonames
WHERE levenshtein(name, 'Sudberg') <= 2


Running query in 'duckdb'

gndUri,name,isPreferred,geonameId,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code
https://d-nb.info/gnd/7557072-5,Kuberg,True,3149168,P,PPL,NO,18,5501,None,None,None
https://d-nb.info/gnd/4305569-2,Surberg,True,2824593,P,PPLA4,DE,02,091,09189,09189148,None
https://d-nb.info/gnd/4118889-5,Sulzberg,True,2824787,P,PPL,DE,02,097,09780,09780140,None
https://d-nb.info/gnd/4609199-3,Sosberg,True,2830910,P,PPLA4,DE,08,00,07135,07135080,None
https://d-nb.info/gnd/4240572-5,Sandberg,False,2841935,P,PPLA4,DE,02,096,09673,09673162,None
https://d-nb.info/gnd/5013486-3,Rurberg,True,2842944,P,PPL,DE,07,053,05334,05334028,None
https://d-nb.info/gnd/4095831-0,Husberg,False,2897203,S,FRM,DE,10,00,01057,01057008,None
https://d-nb.info/gnd/7637702-7,Puberg,True,2985164,P,PPL,FR,44,67,674,67381,None


In [22]:
%%sql
SELECT 
    childId AS geonameId, 
    ANY_VALUE(parentId) AS parentCityId
FROM hierarchy
JOIN geonames AS parent ON hierarchy.parentId = parent.geonameId
JOIN geonames AS child ON hierarchy.childId = child.geonameId
WHERE 
    parent.feature_class = 'P' AND parent.feature_code != 'PPLX' AND
    hierarchy.type != 'ADM'
GROUP BY childId
HAVING COUNT(*) > 1;

Running query in 'duckdb'

geonameId,parentCityId
